# Trabalho Grau B - Reconhecimento de imagem e transfer learning

## Integrantes

- Arthur Schallenberger
- Giovani de Souza
- Leonardo Fronza
- Renan Milech Pereira

---

**Docente:** Prof. Gabriel de Oliveira Ramos

## 2.1 Descrição do Problema e Dataset

### 2.1.1 Descrição do Problema

O problema abordado neste trabalho é a **classificação multiclasse de imagens de bolas esportivas**. Dado um conjunto de imagens coloridas, o objetivo é treinar um modelo capaz de identificar corretamente a qual modalidade esportiva pertence a bola presente na imagem, dentre 15 categorias distintas.

Trata-se de um problema de **visão computacional supervisionada**, em que cada imagem possui um único rótulo associado (classe da bola). O desafio central está na variação visual entre as classes — algumas bolas possuem formas, texturas e padrões muito distintos (como a bola de futebol americano e a de tênis de mesa), enquanto outras apresentam características visuais próximas (como bola de cricket, hockey e tênis), exigindo que o modelo aprenda representações discriminativas e generalizáveis.

A escolha desse domínio é motivada pela disponibilidade de dados rotulados e pela aplicabilidade prática em sistemas de análise de transmissões esportivas, arbitragem automatizada e catalogação de conteúdo multimídia.

---

### 2.1.2 Dataset

O dataset utilizado é o **Sports Ball Image Recognition**, disponível publicamente na plataforma [Kaggle](https://www.kaggle.com/). Ele é composto por imagens JPEG de bolas de 15 modalidades esportivas diferentes, já organizadas em subpastas por classe e divididas entre conjuntos de treino e teste.

#### Estrutura de diretórios

```
archive/
├── train/
│   ├── american_football/
│   ├── baseball/
│   ├── ...
│   └── volleyball/
└── test/
    ├── american_football/
    ├── baseball/
    ├── ...
    └── volleyball/
```

#### Classes e distribuição de imagens

O dataset contém **15 classes**, com a seguinte distribuição por split:

| Classe               | Treino | Teste | Total |
|----------------------|--------|-------|-------|
| american_football    | 384    | 96    | 480   |
| baseball             | 400    | 100   | 500   |
| basketball           | 340    | 86    | 426   |
| billiard_ball        | 646    | 162   | 808   |
| bowling_ball         | 440    | 111   | 551   |
| cricket_ball         | 581    | 146   | 727   |
| football             | 604    | 151   | 755   |
| golf_ball            | 549    | 138   | 687   |
| hockey_ball          | 530    | 133   | 663   |
| hockey_puck          | 390    | 98    | 488   |
| rugby_ball           | 493    | 124   | 617   |
| shuttlecock          | 429    | 108   | 537   |
| table_tennis_ball    | 620    | 156   | 776   |
| tennis_ball          | 490    | 123   | 613   |
| volleyball           | 432    | 109   | 541   |
| **Total**            | **7.328** | **1.841** | **9.169** |

A divisão treino/teste segue uma proporção aproximada de **80%/20%**, padrão comum em benchmarks de visão computacional.

#### Características das imagens

- **Formato:** JPEG (`.jpg`)
- **Resolução:** variada — as imagens originais possuem dimensões heterogêneas (desde 225×225 até resoluções maiores como 1920×1080 e 2048×1152)
- **Canais:** RGB (3 canais de cor)
- **Pré-processamento necessário:** todas as imagens serão redimensionadas para **224×224 pixels** antes de serem alimentadas nos modelos, por ser o formato esperado pelas arquiteturas EfficientNetB0 e MobileNetV2, e também utilizado na CNN própria para padronização

#### Balanceamento das classes

O dataset apresenta um **leve desbalanceamento** entre as classes. A classe com mais amostras (*billiard_ball*) possui 808 imagens, enquanto a menor (*basketball*) possui 426 — uma razão de aproximadamente 1,9×. Esse nível de desbalanceamento é considerado moderado e não exige técnicas agressivas de reamostragem, mas será monitorado durante o treinamento por meio de métricas por classe (precisão, revocação e F1-score).


## 2.2 Análise das Classes e Dados

O conjunto de dados utilizado corresponde ao repositório 'sports-balls-multiclass-image-classification' (Kaggle), organizado em subdiretórios por classe e contendo amostras para treino e teste em `DATASET/archive`.

Visão geral:
- Número de classes: 15.
- Total de imagens (conjunto de treino): 7.328.
- Distribuição por classe: variação moderada entre aproximadamente 340 e 646 imagens por classe, indicando um desbalanceamento leve.
- Resolução média das imagens: aproximadamente 606×509 pixels.
- Integridade: não foram detectadas imagens corrompidas no conjunto de treino analisado.

Organização dos dados:
Os arquivos encontram-se estruturados em subpastas por rótulo (`DATASET/archive/train/<classe>`), facilitando a associação direta imagem→rótulo e a replicabilidade dos experimentos.

Observações qualitativas:
Há variação significativa nas condições de aquisição (fundos, iluminação e escalas), bem como diferenças no tamanho relativo do objeto em relação ao quadro. Algumas classes apresentam similaridades visuais que podem dificultar a discriminação, por exemplo entre `football` e `rugby_ball`, ou entre `hockey_ball` e `hockey_puck`. Essas características implicam maior exigência na capacidade do modelo de extrair representações discriminativas.

Considerações metodológicas:
A heterogeneidade das imagens e as semelhanças inter-classes sugerem a adoção de estratégias de modelagem que privilegiem a generalização e a robustez a variações geométricas e fotométricas. A avaliação deve contemplar medidas por classe (precisão, revocação e F1-score) e análise da matriz de confusão, de modo a evidenciar comportamentos assimétricos entre classes.

Implicações para seleção de modelos e avaliação:
Modelos pré-treinados (por exemplo, arquiteturas amplamente utilizadas em visão computacional) são adequados para exploração inicial do problema, dado o volume de dados e a diversidade visual. Também é pertinente considerar abordagens de fine-tuning e estratégias de regularização que mitiguem overfitting em classes com menor número de amostras. Caso se observe queda de desempenho concentrada em algumas classes, técnicas de ponderação e análise dirigida de falsos positivos devem ser empregadas para diagnosticar as causas subjacentes.

Resumo conclusivo:
O conjunto de dados apresenta qualidade e diversidade suficientes para a investigação de modelos de classificação multiclasses. A análise qualitativa realizada aponta para desafios esperados (variação de aquisição e classes visualmente próximas) que devem ser explicitados no relatório final e considerados na definição do protocolo experimental.


## 2.3 Pré-processamento

O pré-processamento padroniza as imagens antes do treinamento e aplica augmentação para melhorar a generalização dos modelos. As etapas adotadas são descritas a seguir.

### 2.3.1 Redimensionamento

Todas as imagens são redimensionadas para **224 × 224 pixels**, resolução exigida pelas arquiteturas EfficientNetB0 e MobileNetV2 (entrada padrão treinada com ImageNet) e adotada também na CNN própria para uniformizar os experimentos. O redimensionamento é realizado durante o carregamento via `ImageDataGenerator`, sem modificar os arquivos originais.

### 2.3.2 Normalização

A normalização é aplicada de forma específica para cada modelo, pois cada arquitetura foi treinada com uma escala de pixels diferente:

| Modelo         | Função de normalização                             | Intervalo de saída |
|----------------|----------------------------------------------------|--------------------|
| CNN própria    | `rescale = 1 / 255`                                | [0, 1]             |
| EfficientNetB0 | `efficientnet.preprocess_input`                    | Centrado em 0 (ImageNet) |
| MobileNetV2    | `mobilenet_v2.preprocess_input`                    | [−1, 1]            |

O uso das funções `preprocess_input` específicas de cada rede garante compatibilidade com os pesos pré-treinados no ImageNet, evitando degradação de desempenho na fase de *transfer learning*.

### 2.3.3 Divisão Treino / Validação / Teste

O dataset já fornece uma separação explícita entre `train/` e `test/`. Para o conjunto de validação, são reservados **10 %** das amostras de treino via `validation_split`, com semente fixa (`SEED = 42`) para reprodutibilidade:

| Conjunto  | Origem                  | Imagens (aprox.) |
|-----------|-------------------------|-----------------|
| Treino    | `archive/train/` (90 %) | ≈ 6.595          |
| Validação | `archive/train/` (10 %) | ≈ 733            |
| Teste     | `archive/test/`         | 1.841            |

### 2.3.4 Augmentação de Dados

As transformações abaixo são aplicadas **apenas nas imagens de treino**. Os conjuntos de validação e teste recebem somente a normalização, para que a avaliação seja feita com amostras o mais próximas possível das condições reais.

| Transformação       | Parâmetro          | Justificativa                                        |
|---------------------|--------------------|------------------------------------------------------|
| Rotação             | ±15°               | Bolas podem aparecer em qualquer orientação           |
| Deslocamento H / V  | ±10 %              | Variações de enquadramento na captura                |
| Flip horizontal     | Ativado            | Simetria presente na maioria das bolas               |
| Zoom                | ±10 %              | Variação de escala do objeto no quadro               |
| Brilho              | [0,8 – 1,2]        | Condições variadas de iluminação                     |

Essas augmentações introduzem diversidade artificial sem distorcer as características discriminativas das bolas (forma, textura e padrão de cor), contribuindo para reduzir o *overfitting*, especialmente nas classes com menor número de amostras.


## 2.4 Arquitetura das Redes Neurais

Para o trabalho, foram consideradas duas abordagens complementares: uma CNN construída do zero, usada como linha de base, e redes pré-treinadas com *transfer learning*, especialmente EfficientNetB0 e MobileNetV2 (que são redes leves e eficientes e que uma delas será escolhida para o transfer learning).

### 2.4.1 CNN própria

A CNN própria foi pensada para ser simples, estável e fácil de interpretar. A arquitetura proposta segue a lógica de extração progressiva de características:

- **Camada de entrada:** imagens redimensionadas para um formato fixo, como 224 x 224 x 3.
- **Blocos convolucionais:** 3 blocos com convoluções 2D, ativação ReLU e *padding* igual.
- **Pooling:** *MaxPooling2D* após cada bloco convolucional para reduzir dimensionalidade e manter as informações mais relevantes.
- **Regularização:** *Dropout* entre os blocos e antes da saída para reduzir *overfitting*.
- **Classificação final:** camadas densas com *softmax* na saída, uma neurônio por classe.

Uma configuração coerente para essa CNN é:

- Bloco 1: 32 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 2: 64 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 3: 128 filtros, convolução 3 x 3, ReLU, MaxPooling
- *Flatten* ou *GlobalAveragePooling2D*
- *Dense* final com *softmax*

Essa estrutura é suficiente para capturar padrões visuais básicos e serve como referência para comparar com as redes pré-treinadas.

### 2.4.2 Transfer learning

A rede de *transfer learning* escolhida para o experimento principal foi a **EfficientNetB0**, por apresentar bom equilíbrio entre desempenho e custo computacional. A **MobileNetV2** foi mantida como comparação leve, útil quando a prioridade é reduzir parâmetros e acelerar inferência.

A estratégia adotada para a EfficientNetB0 (experimento principal):

1. Carregar os pesos pré-treinados no ImageNet.
2. Congelar a base convolucional nas primeiras etapas.
3. Adicionar uma cabeça de classificação específica para as classes do conjunto de dados.
4. Se necessário, liberar parte das últimas camadas para *fine-tuning*.

### 2.4.3 Justificativa das escolhas

As escolhas arquiteturais foram feitas considerando o tamanho do conjunto de dados e o objetivo de classificação de imagens esportivas:

- A **CNN própria** funciona como baseline e permite avaliar o quanto o problema pode ser resolvido sem conhecimento prévio transferido.
- **EfficientNetB0** tende a oferecer melhor relação entre profundidade, eficiência e generalização, sendo uma boa candidata para maior acurácia.
- **MobileNetV2** é uma alternativa mais leve, com menor custo de processamento, útil para comparação e para cenários com limitação de recursos.
- O uso de **Dropout** e de *MaxPooling* ajuda a controlar o sobreajuste e a reduzir o tamanho das representações intermediárias.
- A ativação **ReLU** é adequada por ser simples, eficiente e amplamente usada em CNNs modernas.

### 2.4.4 Comparação das arquiteturas

| Arquitetura | Número de camadas | Convoluções | Pooling | Dropout | Função de ativação | Vantagem principal |
| --- | --- | --- | --- | --- | --- | --- |
| CNN própria | 3 blocos convolucionais + classificadores | 3 x 3 | MaxPooling2D | Sim | ReLU / Softmax | Baseline simples e interpretável |
| EfficientNetB0 | Backbone pré-treinado + cabeça densa | Convoluções otimizadas pela família EfficientNet | GlobalAveragePooling2D ou pooling implícito | Sim | Swish/ReLU + Softmax | Melhor equilíbrio entre desempenho e custo |
| MobileNetV2 | Backbone pré-treinado + cabeça densa | Convoluções separáveis | GlobalAveragePooling2D | Sim | ReLU6 + Softmax | Menor custo computacional |

### 2.4.5 Diagrama simplificado

```mermaid
flowchart LR
    A[Imagem de entrada\n224 x 224 x 3] --> B{Estratégia}
    B --> C[CNN própria\nConv 3x3 + ReLU\nMaxPooling + Dropout]
    B --> D[EfficientNetB0\nbase congelada + cabeça densa]
    B --> E[MobileNetV2\nbase congelada + cabeça leve]
    C --> F[Softmax\nclassificação]
    D --> F
    E --> F
```

Em resumo, a CNN própria fornece a linha de base do estudo, enquanto EfficientNetB0 e MobileNetV2 representam as melhores alternativas de *transfer learning* para comparar desempenho, robustez e custo de execução.

# 3 CNN própria

Esta seção apresenta a arquitetura CNN desenvolvida "do zero" para o conjunto de dados.

## 3.1 Objetivos
- Propor uma arquitetura simples e eficiente para classificação das 15 classes de bola.
- Fornecer justificativa arquitetural e um resumo do modelo.

## 3.2 Arquitetura proposta
- Entrada: imagens redimensionadas para `224x224x3`.
- Bloco 1: Conv(32,3x3) -> ReLU -> Conv(32,3x3) -> ReLU -> MaxPool(2x2) -> Dropout(0.25)
- Bloco 2: Conv(64,3x3) -> ReLU -> Conv(64,3x3) -> ReLU -> MaxPool(2x2) -> Dropout(0.25)
- Bloco 3: Conv(128,3x3) -> ReLU -> MaxPool(2x2) -> Dropout(0.25)
- Classificador: Flatten -> Dense(256) -> ReLU -> Dropout(0.5) -> Dense(num_classes) -> Softmax

## 3.3 Justificativa arquitetural
- Profundidade moderada para evitar overfitting e manter custo computacional baixo.
- Crescimento de filtros por bloco (32→64→128) para capturar features de complexidade crescente.
- Pares de convoluções nos primeiros blocos inspirados em VGG para maior capacidade representacional.
- MaxPooling reduz dimensionalidade espacial; Dropout controla overfitting.

O código abaixo define a arquitetura e exibe o `model.summary()`.

In [3]:
from tensorflow.keras import layers, models


def build_model(input_shape=(224, 224, 3), num_classes=15, dropout_rate=0.5):
    """Constrói a CNN do zero usada como baseline."""
    model = models.Sequential(name='CNN_Baseline')
    model.add(layers.Input(shape=input_shape))

    # Bloco 1
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Bloco 2
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Bloco 3
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Classificador
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model


# Instanciar e mostrar resumo
model = build_model()
model.summary()

Model: "CNN_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,833,647 (98.55 MB)

 Trainable params: 25,833,647 (98.55 MB)

 Non-trainable params: 0 (0.00 B)

# 3.4 Experimentos dos hiperparâmetros

Plano inicial de varredura de hiperparâmetros para a arquitetura `CNN_Baseline`.

- Otimizadores: `Adam`, `SGD` (momentum=0.9)
- Learning rates: 1e-4, 5e-4, 1e-3, 5e-3
- Batch sizes: 16, 32, 64
- Dropout (fc): 0.25, 0.4, 0.5
- Dropout (conv blocks): 0.15, 0.25, 0.35
- Número de filtros base: base=16 / base=32 (padrão)
- Regularização L2: 0, 1e-4, 1e-3
- Data augmentation: rotações=20, width_shift=0.1, height_shift=0.1, zoom=0.1, horizontal_flip=True
- Épocas: 30, 50, 100 (usar EarlyStopping com `patience=8`)

Protocolo recomendado:
1. Validar configuração base (Adam, lr=1e-3, batch=32, dropout conv=0.25, dropout fc=0.5) por 30 épocas.
2. Rodar buscas em grade reduzidas por bloco de parâmetros.
3. Se necessário, usar pesquisa bayesiana (Optuna) para otimizar múltiplos hiperparâmetros.

Guardar resultados em CSV com: run_id, optimizer, lr, batch_size, dropout_conv, dropout_fc, base_filters, l2, augmentation, val_accuracy, val_loss, epochs_trained.

# 3.5 Pipeline de dados (train / val / test)

Nesta célula definimos os geradores de imagens para treino, validação e teste.
- Redimensionamento para `224x224`
- Normalização para `[0, 1]` (`rescale=1./255`) para a CNN própria
- Augmentação aplicada **apenas** ao conjunto de treino
- `validation_split=0.1` com semente fixa `SEED = 42`

Os geradores seguem o formato esperado por `model.fit()` do Keras.

In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

KAGGLE_DATASET = "samuelcortinhas/sports-balls-multiclass-image-classification"
IN_COLAB = "google.colab" in sys.modules

# ==================================================
# CONFIGURAÇÕES
# ==================================================

MOUNT_DRIVE = True

MODEL_DIR = 'models'
LOG_DIR = 'logs'

# ==================================================
# COLAB SETUP
# ==================================================

if IN_COLAB:
    from google.colab import drive, userdata

    # ----------------------------------------------
    # MONTAR GOOGLE DRIVE
    # ----------------------------------------------

    if MOUNT_DRIVE:
        drive.mount('/content/drive')

        DRIVE_ROOT = '/content/drive/MyDrive'
        DRV_ROOT = os.path.join(DRIVE_ROOT, 'ImageRecognition')

        os.makedirs(DRV_ROOT, exist_ok=True)

        MODEL_DIR = os.path.join(DRV_ROOT, 'models')
        LOG_DIR = os.path.join(DRV_ROOT, 'logs')

        print(f'Drive montado.')
        print(f'Artefatos serão salvos em: {DRV_ROOT}')

    else:
        print('Drive não montado.')

    # ----------------------------------------------
    # CONFIGURAR KAGGLE API
    # ----------------------------------------------

    kaggle_dir = Path('/root/.kaggle')
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    kaggle_json = kaggle_dir / 'kaggle.json'

    username = None
    key = None

    # Tenta pegar dos Colab Secrets
    try:
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
    except Exception:
        pass

    # Fallback: variáveis de ambiente
    if (not username or not key):
        username = os.environ.get('KAGGLE_USERNAME')
        key = os.environ.get('KAGGLE_KEY')

    if not username or not key:
        raise RuntimeError(
            'Credenciais do Kaggle não encontradas.\n'
            'Configure KAGGLE_USERNAME e KAGGLE_KEY nos Secrets do Colab.'
        )

    payload = {
        'username': username,
        'key': key
    }

    kaggle_json.write_text(
        json.dumps(payload),
        encoding='utf-8'
    )

    os.chmod(kaggle_json, 0o600)

    print('Kaggle API configurada.')

    # ----------------------------------------------
    # INSTALAR KAGGLE
    # ----------------------------------------------

    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'kaggle'
    ])

    # ----------------------------------------------
    # DOWNLOAD DATASET
    # ----------------------------------------------

    DATASET_PATH_CHECK = Path('/content/train')

    if not DATASET_PATH_CHECK.exists():

        print('Dataset não encontrado.')
        print('Baixando dataset do Kaggle...')

        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()

        api.dataset_download_files(
            KAGGLE_DATASET,
            path='/content',
            unzip=True
        )

        print('Dataset baixado com sucesso!')

    else:
        print('Dataset já existe. Download ignorado.')

# ==================================================
# DETECTAR DATA_ROOT
# ==================================================

candidate_roots = [
    '/content/archive',
    '/content/dataset/archive',
    '/content/dataset',
    '/content',
    'dataset/archive',
    'dataset'
]

DATA_ROOT = None

for root in candidate_roots:

    train_dir = os.path.join(root, 'train')
    test_dir = os.path.join(root, 'test')

    if os.path.isdir(train_dir) and os.path.isdir(test_dir):
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        'Não foi possível localizar as pastas train/ e test/.'
    )

# ==================================================
# GARANTIR DIRETÓRIOS
# ==================================================

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ==================================================
# OUTPUT
# ==================================================

print(f'DATA_ROOT detectado: {DATA_ROOT}')
print(f'MODEL_DIR: {MODEL_DIR}')
print(f'LOG_DIR: {LOG_DIR}')

In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Usa DATA_ROOT detectado na célula de setup; fallback para execução local
if 'DATA_ROOT' not in globals():
    candidate_roots = ['dataset/archive', 'dataset']
    for root in candidate_roots:
        if os.path.isdir(os.path.join(root, 'train')) and os.path.isdir(os.path.join(root, 'test')):
            DATA_ROOT = root
            break
    else:
        raise FileNotFoundError('DATA_ROOT não definido e dataset não encontrado em dataset/archive ou dataset.')

TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
TEST_DIR = os.path.join(DATA_ROOT, 'test')

# Gerador de treino com augmentação e validation_split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
    validation_split=0.1
)

# Gerador de validação (somente rescale)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.1)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED
)

# Gerador de teste (somente rescale)
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('DATA_ROOT:', DATA_ROOT)
print('Classes:', train_generator.class_indices)
print('Train samples:', train_generator.samples)
print('Val samples:', val_generator.samples)
print('Test samples:', test_generator.samples)

train_steps = train_generator.samples // BATCH_SIZE
val_steps = val_generator.samples // BATCH_SIZE
test_steps = (test_generator.samples + BATCH_SIZE - 1) // BATCH_SIZE
print('Steps (train/val/test):', train_steps, val_steps, test_steps)

# Sanity check: pegar um batch e verificar shapes
x_batch, y_batch = next(train_generator)
print('Batch shapes:', x_batch.shape, y_batch.shape)


# 3.6 Compilação do modelo e callbacks

Nesta célula compilamos o modelo (`Adam`, `learning_rate=1e-3`) e definimos callbacks úteis para treino:
- `ModelCheckpoint` (salva o melhor modelo)
- `EarlyStopping` (parada antecipada com restauração dos melhores pesos)
- `ReduceLROnPlateau` (reduz LR quando a métrica estaciona)
- `TensorBoard` (logs para visualização)
- `CSVLogger` (registra métricas em CSV)

Também calculamos `class_weights` a partir do gerador de treino (se `sklearn` estiver disponível usa `compute_class_weight`, caso contrário utiliza um fallback).

In [ ]:
from tensorflow import keras
from tensorflow.keras import optimizers, callbacks
import os

# Hyperparams
LR = 1e-3
EPOCHS = 30

# Build and compile model
model = build_model()
optimizer = optimizers.Adam(learning_rate=LR)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Make directories (MODEL_DIR and LOG_DIR are defined in the setup cell; fall back to local if not present)
if 'MODEL_DIR' not in globals():
    MODEL_DIR = 'models'
if 'LOG_DIR' not in globals():
    LOG_DIR = 'logs'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Callbacks
checkpoint_path = os.path.join(MODEL_DIR, 'best_cnn.h5')
checkpoint_cb = callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_loss')
earlystop_cb = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr_cb = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)
tensorboard_cb = callbacks.TensorBoard(log_dir=os.path.join(LOG_DIR, 'fit'))
csv_logger = callbacks.CSVLogger(os.path.join(LOG_DIR, 'training_log.csv'))

callbacks_list = [checkpoint_cb, earlystop_cb, reduce_lr_cb, tensorboard_cb, csv_logger]

# Compute class weights
try:
    import numpy as np
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(train_generator.classes)
    cw = compute_class_weight('balanced', classes=classes, y=train_generator.classes)
    class_weights = dict(enumerate(cw))
except Exception as e:
    print('sklearn not available or error computing class weights, using fallback:', e)
    import numpy as np
    counts = np.bincount(train_generator.classes)
    total = counts.sum()
    class_weights = {i: float(total) / (len(counts) * c) for i, c in enumerate(counts)}

print('Class weights:', class_weights)

# Summary
model.summary()
print('Callbacks:', [type(cb).__name__ for cb in callbacks_list])

# Remover os comentarios para rodar o treinamento:
history = model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=val_generator,
    validation_steps=val_steps,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    class_weight=class_weights,
)
